**1. 依序生成各個系統的運算結果**

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from smart_book_seeker import ConversationManager
load_dotenv()

ES_API_URL = os.getenv("ES_API_URL")
ES_API_KEY = os.getenv("ES_API_KEY")
header = {
    "Authorization": f"Apikey {ES_API_KEY}",
    "Content-Type": "application/json"
}
for i in range(1, 101):
    # Ground Truth
    filename = f"../results/ground_truth/{i:03}.json"
    with open(filename, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)
    print(f"[{i:03}/100] Ground Truth ✓")

    # Baseline
    baseline = {"user_book_needs": ground_truth["user_book_needs"], "books": []}
    post_data = {
        "query": {
            "simple_query_string": {
                "query": ground_truth["user_book_needs"]
            }
        }
    }
    response = requests.get(f"{ES_API_URL}/books/_search?size=10", headers=header, json=post_data)
    for book in response.json()["hits"]["hits"]:
        baseline["books"].append({
            "id": book["_source"]["id"],
            "title": book["_source"]["title"]
        })
    filename = f"../results/baseline/{i:03}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(baseline, f, indent=2, ensure_ascii=False)
    print(f"[{i:03}/100] Baseline ✓")

    # Smart Book Seeker
    sbs = {"user_book_needs": ground_truth["user_book_needs"], "books": []}
    cm = ConversationManager(strategy="iterative_top_k")
    max_retries = 10
    retry_count = 0
    while retry_count < max_retries:
        try:
            for response in cm.discuss(book_search_needs=sbs["user_book_needs"]):
                pass
            break
        except ValueError as e:
            retry_count += 1
            print(f"[{i:03}/100] SBS ✗ ({e}, {retry_count}/{max_retries})")
            if retry_count == max_retries:
                print(f"[{i:03}/100] SBS ✗ (Max retries reached)")
                break
    if retry_count >= max_retries:
        print("Exit due to max retries")
        break
    for book in cm.user_agent.environment.current_books:
        sbs["books"].append({
            "id": book.id,
            "title": book.title
        })
    filename = f"../results/smart_book_seeker/{i:03}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(sbs, f, indent=2, ensure_ascii=False)
    print(f"[{i:03}/100] Smart Book Seeker ✓")

**2. 計算成績**

In [ ]:
import json
import numpy as np

# load results
result_ground_truth = []
result_baseline = []
result_sbs = []
for i in range(1, 101):
    filename = f"../results/ground_truth/{i:03}.json"
    with open(filename, "r", encoding="utf-8") as f:
        result_ground_truth.append(json.load(f))
    filename = f"../results/baseline/{i:03}.json"
    with open(filename, "r", encoding="utf-8") as f:
        result_baseline.append(json.load(f))
    filename = f"../results/smart_book_seeker/{i:03}.json"
    with open(filename, "r", encoding="utf-8") as f:
        result_sbs.append(json.load(f))

# calculate scores
score_baseline = []
score_sbs = []
for i, result in enumerate(result_baseline):
    score_baseline.append(0)
    score_sbs.append(0)
    for book in result_baseline[i]["books"]:
        for ans_book in result_ground_truth[i]["books"]:
            if book["id"] == ans_book["id"]:
                score_baseline[i] += 10
    for book in result_sbs[i]["books"]:
        for ans_book in result_ground_truth[i]["books"]:
            if book["id"] == ans_book["id"]:
                score_sbs[i] += 10

# calculate average, variance, median, std
avg_baseline = sum(score_baseline) / len(score_baseline)
var_baseline = np.var(score_baseline)
median_baseline = np.median(score_baseline)
std_baseline = np.std(score_baseline)
avg_sbs = sum(score_sbs) / len(score_sbs)
var_sbs = np.var(score_sbs)
median_sbs = np.median(score_sbs)
std_sbs = np.std(score_sbs)

print(f"Baseline: avg={avg_baseline:.2f}, var={var_baseline:.2f}, median={median_baseline:.2f}, std={std_baseline:.2f}")
print(f"SBS:      avg={avg_sbs:.2f}, var={var_sbs:.2f}, median={median_sbs:.2f}, std={std_sbs:.2f}")

**3. 畫圖**

In [ ]:
import matplotlib.pyplot as plt

data   = [score_baseline, score_sbs]
labels = ['Baseline', 'SBS-AARS']

plt.figure(figsize=(6, 4), dpi=300)
box = plt.boxplot(
    data,
    patch_artist=True,
    boxprops=dict(color='black'),
    medianprops=dict(color='black', linewidth=1.5),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
hatches = ['', '///']
for patch, hatch in zip(box['boxes'], hatches):
    patch.set_facecolor('white')
    patch.set_hatch(hatch)
    patch.set_edgecolor('black')
plt.xticks([1, 2], labels)
plt.ylabel('Precision Rate (%)')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
# plt.title('Baseline v.s. SBS-AARS')
# legend_elements = [
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[0],
#           label=f'Baseline (Average = {avg1:.2f}%)'),
#     Patch(facecolor='white', edgecolor='black', hatch=hatches[1],
#           label=f'Smart Book Seeker (Average = {avg2:.2f}%)')
# ]
# plt.legend(handles=legend_elements, loc='upper left')

plt.savefig('../logs/boxplot.pdf')
plt.show()
plt.close()